# Multi-Dataset Benchmark Runner

Run CPU/GPU bitset benchmarks directly from Jupyter and save outputs into `data/`.
After this notebook, open `plot_dataset_size.ipynb` to visualize results.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import os
import sys

import pandas as pd

def _resolve_profiling_dir() -> Path:
    env_path = os.getenv('PROFILE_TELCO_DIR')
    if env_path:
        candidate = Path(env_path).expanduser().resolve()
        if (candidate / 'profile_telco_bitset.py').exists():
            return candidate

    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]
    for root in search_roots:
        candidate = root / 'notebooks' / 'profiling'
        if (candidate / 'profile_telco_bitset.py').exists():
            return candidate

    if (cwd / 'profile_telco_bitset.py').exists():
        return cwd
    if (cwd.parent / 'profile_telco_bitset.py').exists():
        return cwd.parent

    raise FileNotFoundError(
        'Could not find notebooks/profiling/profile_telco_bitset.py. '
        'Set PROFILE_TELCO_DIR or run the notebook from the repository tree.'
    )

profiling_dir = _resolve_profiling_dir()
current_dir = profiling_dir / 'dataset_size'
if not current_dir.exists():
    raise FileNotFoundError(f'Dataset-size directory not found: {current_dir}')

if str(profiling_dir) not in sys.path:
    sys.path.insert(0, str(profiling_dir))

from profile_telco_bitset import run_profile

print('current_dir:', current_dir)
print('profiling_dir:', profiling_dir)


In [ ]:
# Configuration
DATASETS = ['telco', 'adult', 'census_income']
RUNS_PER_SETTING = 10
MODES = ['cpu', 'gpu']  # allowed: 'cpu', 'gpu'
MAX_GPU_MEM_MB = None
GPU_BATCH_SIZE = None
TAG = ''  # optional string, e.g. 'serverA'

OUTPUT_DIR = current_dir / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATASETS:', DATASETS)
print('RUNS_PER_SETTING:', RUNS_PER_SETTING)
print('MODES:', MODES)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
records = []
for mode in MODES:
    if mode not in {'cpu', 'gpu'}:
        raise ValueError(f'Unsupported mode: {mode}')
    use_gpu = mode == 'gpu'

    for dataset in DATASETS:
        for run_index in range(RUNS_PER_SETTING):
            print(f'mode={mode} dataset={dataset} run={run_index + 1}/{RUNS_PER_SETTING}')
            result = run_profile(
                use_gpu=use_gpu,
                dataset=str(dataset),
                repeat_factor=1,
                max_gpu_mem_mb=MAX_GPU_MEM_MB,
                gpu_batch_size=GPU_BATCH_SIZE,
                verbose=False,
            )
            records.append({
                'timestamp_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                'dataset_key': str(dataset),
                'mode_key': mode,
                'repeat_factor': 1,
                'run_index': int(run_index),
                **result,
            })

df_runs = pd.DataFrame(records)
print(f'Total records: {len(df_runs)}')
df_runs.head()


In [ ]:
if df_runs.empty:
    raise RuntimeError('No records to save.')

ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
suffix = f'_{TAG}' if TAG else ''
jsonl_path = OUTPUT_DIR / f'dataset_size_runs{suffix}_{ts}.jsonl'
csv_path = OUTPUT_DIR / f'dataset_size_runs{suffix}_{ts}.csv'
latest_jsonl = OUTPUT_DIR / 'dataset_size_runs_latest.jsonl'
latest_csv = OUTPUT_DIR / 'dataset_size_runs_latest.csv'

with jsonl_path.open('w', encoding='utf-8') as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=True) + '\n')

fieldnames = list(records[0].keys())
with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

latest_jsonl.write_text(jsonl_path.read_text(encoding='utf-8'), encoding='utf-8')
latest_csv.write_text(csv_path.read_text(encoding='utf-8'), encoding='utf-8')

print('Saved JSONL:', jsonl_path)
print('Saved CSV:  ', csv_path)
print('Updated:    ', latest_jsonl)
print('Updated:    ', latest_csv)


In [ ]:
summary = (
    df_runs.groupby(['dataset_key', 'mode'], as_index=False)
    .agg(
        runs=('elapsed_seconds', 'count'),
        mean_s=('elapsed_seconds', 'mean'),
        median_s=('elapsed_seconds', 'median'),
        std_s=('elapsed_seconds', lambda x: float(x.std(ddof=1)) if len(x) > 1 else 0.0),
        rules_min=('rule_count', 'min'),
        rules_max=('rule_count', 'max'),
    )
    .sort_values(['dataset_key', 'mode'])
)
summary
